###### Bronze ingestion pipeline
###### Raw JSON -> Delta Bronze tables (explicit DB.table to avoid Fabric catalog ambiguity)

In [6]:
import sys
SRC_PATH = "/lakehouse/default/Files/yelp/src"
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print("SRC_PATH OK:", SRC_PATH)


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 8, Finished, Available, Finished)

SRC_PATH OK: /lakehouse/default/Files/yelp/src


In [7]:
from datetime import datetime
from utils.config import RAW_FILES, BRONZE_TABLES
from bronze.ingest_json_to_bronze import ingest_json_to_bronze

print("imports OK")


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 9, Finished, Available, Finished)

imports OK


In [8]:
RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

DB = spark.sql("SELECT current_database()").collect()[0][0]
print("RUN_ID:", RUN_ID)
print("DB:", DB)


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 10, Finished, Available, Finished)

RUN_ID: 20260121_184259
DB: chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u


In [9]:

BRONZE_TABLES_DB = {
    k: (v if "." in v else f"{DB}.{v}")
    for k, v in BRONZE_TABLES.items()
}

print(BRONZE_TABLES_DB)


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 11, Finished, Available, Finished)

{'business': 'chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.business_bronze', 'review': 'chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.review_bronze', 'user': 'chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.user_bronze', 'tip': 'chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.tip_bronze', 'checkin': 'chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.checkin_bronze'}


In [10]:
def log(stage: str, status: str, msg: str):
    ts = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{stage}][{status:<6}] ts={ts} run_id={RUN_ID} {msg}")

mode = "overwrite"
keys = ["business", "review", "user", "checkin", "tip"]

ok, failed = 0, 0

for k in keys:
    src = RAW_FILES[k]
    tgt = BRONZE_TABLES_DB[k]
    try:
        log("BRONZE", "START", f"{k:<8} mode={mode} table={tgt} src={src}")
        ingest_json_to_bronze(
            spark=spark,
            raw_path=src,
            bronze_table=tgt,
            mode=mode,
            batch_id=RUN_ID
        )
        ok += 1
        log("BRONZE", "DONE",  f"{k:<8} table={tgt}")
    except Exception as e:
        failed += 1
        log("BRONZE", "FAIL",  f"{k:<8} error={type(e).__name__}: {e}")
        raise

log("BRONZE", "SUMMARY", f"ok={ok} failed={failed}")


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 12, Finished, Available, Finished)

[BRONZE][START ] ts=2026-01-21 18:43:01 run_id=20260121_184259 business mode=overwrite table=chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.business_bronze src=Files/yelp/raw/yelp_business.json
[BRONZE][DONE  ] ts=2026-01-21 18:43:18 run_id=20260121_184259 business table=chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.business_bronze
[BRONZE][START ] ts=2026-01-21 18:43:18 run_id=20260121_184259 review   mode=overwrite table=chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.review_bronze src=Files/yelp/raw/yelp_review.json
[BRONZE][DONE  ] ts=2026-01-21 18:43:53 run_id=20260121_184259 review   table=chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.review_bronze
[BRONZE][START ] ts=2026-01-21 18:43:53 run_id=20260121_184259 user     mode=overwrite table=chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.user_bronze src=Files/yelp/raw/yelp_user.

In [11]:
for k, t in BRONZE_TABLES_DB.items():
    cnt = spark.table(t).count()
    print(f"{k:<8} -> {t:<40} rows={cnt}")


StatementMeta(, 397e6649-38da-429c-84ab-e6bfd18bade0, 13, Finished, Available, Finished)

business -> chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.business_bronze rows=150346
review   -> chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.review_bronze rows=6990280
user     -> chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.user_bronze rows=1987897
tip      -> chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.tip_bronze rows=908915
checkin  -> chimcobldhq2amb5dho2qp31ehgiqpbectkmspb5e9kmspp5f5imos2vdhgmmpb8dtqn6p95chh6u.checkin_bronze rows=131930
